# Senate Stage 1: a lagged Senate-performance predictor (ablation)

Does a state's recent Senate over- or under-performance against fundamentals persist? For each race:

$$r_{s,t} = M^{actual}_{s,t} - \widehat M^{S1}_{s,t}, \qquad L_{s,t} = 0.75\,r_{s,\text{recent}} + 0.25\,r_{s,\text{previous}}$$

**S1 + L** adds L to production S1 (lean, generic ballot, incumbency fixed at +2.5) and estimates one persistence
coefficient, by the same equal-year WLS. Production S1, the blend weights and the forecasts are unchanged. Code:
`src/senate/regression/lagged.py`.

**Rules:**
- **No leakage:** each residual is S1's rolling out-of-sample error, from a fit on the years before that race only.
  The first residuals are for 2010, because a fit on 2006 alone can't separate the generic ballot from the intercept.
- **Lags:** the state's two most recent *earlier* Senate state-years, with both classes and specials counted. A
  same-year pair of races counts as one election, using their mean residual. A same-day special is never a lag.
- **Missing lags:** if either lag's residual is missing (before 2010, or an imputed, third-party or excluded race),
  L is missing. It's never set to 0, and an older election isn't substituted. Those races keep S1's prediction
  (`lag_fallback`).
- **Backtest years:** L exists from 2014, and S1 + L needs two training years with L, so the backtest covers
  2018–2024.
- **Control:** S1 + L can only train on races with L, which means from 2014, while S1 trains from 2006. "S1 (same
  rows)" refits S1 on exactly S1 + L's rows, so the effect of L is separated from the effect of the shorter sample.
- **Neutral variant, "S1 + L (neutral)":** a missing L is set to 0, an explicit no-signal value flagged
  `lag_neutral`, instead of excluding the race. It trains on every observed race from 2006, the same rows as
  production S1, and predicts every race, so its comparison with S1 is matched. The stored `lagged_performance`
  stays missing; the 0 is applied only inside the fit.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
import senate.regression as sr
from senate import stage1, stage2
from senate.regression import lagged

pd.set_option("display.width", 200)
EVAL_YEARS = [2018, 2020, 2022, 2024]
S1 = stage1.SENATE_S1
data = sr.load_data()

residuals = lagged.oos_residuals(stage1, S1, data, range(lagged.FIRST_RESIDUAL_YEAR, 2026, 2))
state_years = lagged.state_year_residuals(data[data["year"] <= 2024], residuals)
lags = lagged.lagged_performance(data, state_years, weights=lagged.WEIGHTS)
data_l = data.merge(lags[[*sr.KEY, lagged.LAG, "lag_available"]], on=sr.KEY, how="left", validate="one_to_one")

## 1. Coverage

In [2]:
display(Markdown("**Races with L, by year** (all races in `senate_incumbency.csv`)"))
display(lags.groupby("year")["lag_available"].agg(with_L="sum", races="size").T)
display(Markdown("**Why L is missing**"))
lags["lag_missing_reason"].dropna().str.replace(r"^\d{4}: ", "", regex=True).value_counts().rename("races").to_frame()

**Races with L, by year** (all races in `senate_incumbency.csv`)

year,2006,2008,2010,2012,2014,2016,2018,2020,2022,2024,2026
with_L,0,0,0,0,3,29,34,32,32,32,32
races,33,35,37,33,36,34,35,35,35,34,35


**Why L is missing**

,races
lag_missing_reason,
before 2010 (no out-of-sample fundamentals),65
only 1 earlier Senate election(s) in the data,53
only 0 earlier Senate election(s) in the data,50
"a race without an observed D-vs-R residual (imputed, third-party or excluded)",20


In [3]:
display(Markdown("**Example lags** (2024)"))
lags[lags["year"] == 2024].head(8).round(2)

**Example lags** (2024)

,year,state,class,recent_year,previous_year,r_recent,r_previous,lagged_performance,lag_missing_reason,lag_available
313,2024,AZ,1,2022.0,2020.0,6.61,1.86,5.42,NaN,True
314,2024,CA,1,2022.0,2018.0,-3.83,NaN,NaN,2018: a race without an observed D-vs-R residu...,False
315,2024,CT,1,2022.0,2018.0,-2.37,-3.25,-2.59,NaN,True
316,2024,DE,1,2020.0,2018.0,1.07,0.35,0.89,NaN,True
317,2024,FL,1,2022.0,2018.0,-7.91,-8.10,-7.95,NaN,True
318,2024,HI,1,2022.0,2018.0,17.18,-3.09,12.12,NaN,True
319,2024,IN,1,2022.0,2018.0,1.19,2.79,1.59,NaN,True
320,2024,MA,1,2020.0,2018.0,-1.23,-11.28,-3.75,NaN,True


## 2. Fits and the persistence coefficient

In [4]:
s1_pred = pd.concat([r.predictions for r in stage1.rolling_predictions(S1, eval_years=EVAL_YEARS, data=data).values()],
                    ignore_index=True)
args = dict(data=data_l, s1_formula=S1.formula, fixed=dict(S1.fixed), s1_predictions=s1_pred, eval_years=EVAL_YEARS)
lag_pred, lag_coef = lagged.fit_predict_lagged(**args, with_lag=True)
ctrl_pred, ctrl_coef = lagged.fit_predict_lagged(**args, with_lag=False)
neutral_pred, neutral_coef = lagged.fit_predict_lagged(**args, with_lag=True, missing_lag="neutral")
PREDICTIONS = {"S1": s1_pred, "S1 (same rows)": ctrl_pred, "S1 + L": lag_pred, "S1 + L (neutral)": neutral_pred}
display(Markdown("**S1 + L coefficients by window** (`lagged_performance` is the persistence coefficient)"))
display(lag_coef)
display(Markdown("**S1 (same rows) coefficients by window**"))
display(ctrl_coef)
display(Markdown("**S1 + L (neutral) coefficients by window** (trained on every observed race; `n_train_with_L` have "
                 "a real L, the rest L = 0)"))
neutral_coef

**S1 + L coefficients by window** (`lagged_performance` is the persistence coefficient)

,Intercept,partisan_lean,gen_margin_raw,lagged_performance,inc_signed,n_train,n_train_with_L,train_years
eval_year,,,,,,,,
2018,-2.918364,1.145414,1.79145,0.326735,2.5,31,31,2014-2016
2020,-0.521462,1.013483,1.242506,0.096185,2.5,65,65,2014-2018
2022,-0.571442,1.013357,1.088918,0.082408,2.5,96,96,2014-2020
2024,1.112236,0.998695,0.831164,0.043716,2.5,127,127,2014-2022


**S1 (same rows) coefficients by window**

,Intercept,partisan_lean,gen_margin_raw,inc_signed,n_train,n_train_with_L,train_years
eval_year,,,,,,,
2018,-0.164594,1.066661,0.300707,2.5,31,31,2014-2016
2020,0.315094,1.004898,1.086866,2.5,65,65,2014-2018
2022,0.145164,1.006609,0.957944,2.5,96,96,2014-2020
2024,1.303742,0.997578,0.789991,2.5,127,127,2014-2022


**S1 + L (neutral) coefficients by window** (trained on every observed race; `n_train_with_L` have a real L, the rest L = 0)

,Intercept,partisan_lean,gen_margin_raw,lagged_performance,inc_signed,n_train,n_train_with_L,train_years
eval_year,,,,,,,,
2018,1.564085,0.985278,0.854889,0.462803,2.5,198,31,2006-2016
2020,1.590079,0.951381,0.880695,0.136443,2.5,232,65,2006-2018
2022,1.42247,0.961413,0.854194,0.112614,2.5,266,96,2006-2020
2024,1.709898,0.964507,0.83193,0.13773,2.5,299,127,2006-2022


## 3. Out-of-sample comparison, 2018–2024

- **Panels:** `all` is every scored race. Races without L use S1's own prediction under S1 + L, and the model with
  L = 0 under S1 + L (neutral). `with L` is the races that have L, the cleanest test.
- **Stage 2:** the blend uses the current inputs and the fixed weights, unchanged.

In [5]:
polls, ratings = stage2.load_inputs()
blends = {name: stage2.blend(p, polls, ratings) for name, p in PREDICTIONS.items()}
with_l = set(map(tuple, lag_pred.loc[~lag_pred["lag_fallback"], sr.KEY].to_numpy()))
COLUMNS = {"fundamentals": "margin_fundamentals", "blend": "prediction"}
SHOW = ["n", "mae", "bias", "ex_ante_n", "ex_ante_mae", "accuracy_pct", "ex_ante_accuracy_pct"]


def panel(frame, name):
    return frame if name == "all" else frame[[tuple(k) in with_l for k in frame[sr.KEY].to_numpy()]]


for name in ["with L", "all"]:
    display(Markdown(f"### Panel: {name}"))
    display(pd.concat({m: stage2.score_table(panel(b, name), data, COLUMNS) for m, b in blends.items()},
                      names=["model", "stage"]).swaplevel().sort_index(level=0)[SHOW].round(2))

### Panel: with L

n   mae  bias  ex_ante_n  ex_ante_mae  accuracy_pct  ex_ante_accuracy_pct
stage        model                                                                                          
blend        S1                127.0  4.14 -1.11       39.0         3.13         92.91                 84.62
             S1 (same rows)    127.0  4.44 -0.04       39.0         3.32         92.13                 82.05
             S1 + L            127.0  4.53 -0.94       39.0         3.19         92.91                 84.62
             S1 + L (neutral)  127.0  4.28 -0.98       39.0         3.02         92.91                 84.62
fundamentals S1                127.0  5.30  0.04       39.0         3.97         92.91                 84.62
             S1 (same rows)    127.0  6.31  2.44       39.0         5.20         89.76                 74.36
             S1 + L            127.0  6.41  0.34       39.0         4.49         90.55                 76.92
             S1 + L (neutral)  127.0  5.78  0.34       39.0         4.32         91.34                 79.49

### Panel: all

n   mae  bias  ex_ante_n  ex_ante_mae  accuracy_pct  ex_ante_accuracy_pct
stage        model                                                                                          
blend        S1                134.0  4.21 -1.33       39.0         3.13         93.28                 84.62
             S1 (same rows)    134.0  4.49 -0.32       39.0         3.32         92.54                 82.05
             S1 + L            134.0  4.58 -1.17       39.0         3.19         93.28                 84.62
             S1 + L (neutral)  134.0  4.34 -1.22       39.0         3.02         93.28                 84.62
fundamentals S1                134.0  5.26 -0.15       39.0         3.97         93.28                 84.62
             S1 (same rows)    134.0  6.23  2.13       39.0         5.20         90.30                 74.36
             S1 + L            134.0  6.32  0.14       39.0         4.49         91.04                 76.92
             S1 + L (neutral)  134.0  5.73  0.13       39.0         4.32         91.79                 79.49

### Blend: polled vs unpolled races (panel: all)

In [6]:
for polled in [True, False]:
    display(Markdown(f"**{'Polled' if polled else 'Unpolled'}**"))
    display(pd.concat({m: stage2.score_table(b[b["polling_available"] == polled], data, {"blend": "prediction"})
                       for m, b in blends.items()}, names=["model"])[SHOW].round(2))

**Polled**

,,n,mae,bias,ex_ante_n,ex_ante_mae,accuracy_pct,ex_ante_accuracy_pct
model,,,,,,,,
S1,blend,87.0,4.00,-1.55,36.0,3.19,89.66,83.33
S1 (same rows),blend,87.0,4.29,-0.60,36.0,3.37,88.51,80.56
S1 + L,blend,87.0,4.17,-1.50,36.0,3.20,89.66,83.33
S1 + L (neutral),blend,87.0,4.08,-1.41,36.0,3.04,89.66,83.33


**Unpolled**

,,n,mae,bias,ex_ante_n,ex_ante_mae,accuracy_pct,ex_ante_accuracy_pct
model,,,,,,,,
S1,blend,47.0,4.61,-0.93,3.0,2.52,100.0,100.0
S1 (same rows),blend,47.0,4.87,0.18,3.0,2.79,100.0,100.0
S1 + L,blend,47.0,5.34,-0.57,3.0,3.09,100.0,100.0
S1 + L (neutral),blend,47.0,4.83,-0.87,3.0,2.81,100.0,100.0


### By year (MAE, panel: with L)

In [7]:
by_year = {}
for m, b in blends.items():
    b = panel(b, "with L")
    for stage, column in COLUMNS.items():
        by_year[(stage, m)] = (b["actual"] - b[column]).abs().groupby(b["year"]).mean()
pd.DataFrame(by_year).T.round(2)

,year,2018,2020,2022,2024
fundamentals,S1,6.39,5.25,5.31,4.14
blend,S1,3.38,5.24,4.23,3.81
fundamentals,S1 (same rows),9.59,5.28,5.98,4.09
blend,S1 (same rows),4.41,5.20,4.38,3.77
fundamentals,S1 + L,9.64,5.33,6.36,3.99
blend,S1 + L,4.55,5.30,4.53,3.73
fundamentals,S1 + L (neutral),8.53,5.23,5.16,3.94
blend,S1 + L (neutral),3.86,5.37,4.15,3.79


## Conclusion

**Persistence is weak, and neither version improves on production S1 overall.** Production is unchanged.

**Strict S1 + L** (races without L excluded from training):
- **The persistence coefficient fades:** 0.33, 0.10, 0.08, then 0.04 by the 2024 window.
- **It's worse than S1 in both stages** (races with L): fundamentals MAE 6.41 vs 5.30, blend 4.53 vs 4.14. Most of the
  loss comes from training only from 2014. Against the matched control (S1 on the same rows), L's own effect is mixed.

**Neutral S1 + L** (missing L = 0; trains on the same rows as S1, so the comparison is matched):
- **The persistence coefficient** is 0.46 in the 2018 window (only 31 training races have a real L), then settles at
  0.11–0.14.
- **The blend splits:** slightly better on competitive races, slightly worse overall (all 134 scored races).

  | | S1 | S1 + L (neutral) |
  |---|---|---|
  | Fundamentals MAE / ex-ante | 5.26 / 3.97 | 5.73 / 4.32 |
  | Blend MAE / ex-ante | 4.21 / 3.13 | 4.34 / **3.02** |
  | Blend, polled / unpolled MAE | 4.00 / 4.61 | 4.08 / 4.83 |
  | Blend winners called | 93.3% | 93.3% |

- **The loss is concentrated in 2018:** the first window's 0.46 coefficient overfits (fundamentals 8.53 vs 6.39).
  In later windows the neutral version is level or slightly ahead: fundamentals 5.16 vs 5.31 (2022) and 3.94 vs 4.14
  (2024), blend 4.15 vs 4.23 and 3.79 vs 3.81.
- **Reading:** a persistence coefficient of about 0.1 is plausible once enough lagged races accumulate, but the gain
  is a few hundredths of a point and is within noise over 4 test years. Not worth adopting now. It would be worth
  re-checking after 2026, when another cycle of lags exists.

**Limitations:**
- 4 test years. Residuals start in 2010, because there's no leakage-free fit before then.
- The neutral 0 treats "no lag" as "no persistence", a modeling assumption rather than an observation; those races are
  flagged `lag_neutral`.
- The 75/25 weights were fixed in advance, not tuned.
